In [ ]:
import pandas as pd
df = pd.read_csv("/content/Demand Forecasting Dataset.csv")

In [ ]:
df.head()

## Data Preprocessing and Feature Alignment

In [ ]:
# Convert 'Date' column to datetime objects
df['Date'] = pd.to_datetime(df['Date'])

# Sort data by 'Store ID', 'Product ID', and 'Date' to ensure proper time-series ordering
df = df.sort_values(by=['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# Identify categorical columns to be encoded (excluding 'Store ID' and 'Product ID' to keep them for grouping)
cols_to_encode = ['Category', 'Region', 'Weather Condition', 'Seasonality']

# Apply one-hot encoding to specified categorical variables
df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)

display(df_encoded.head())

## Discrete Wavelet Decomposition (DWT) and Feature Alignment

In [ ]:
# Install pywt if not already installed
!pip install pywt

import numpy as np
import pywt
import pandas as pd

# Function to downsample a feature series to a target length by averaging segments
def downsample_features_simple(feature_series, target_len):
    if len(feature_series) == 0 or target_len == 0:
        return np.array([])

    resampled_features = []
    step = len(feature_series) / target_len
    for i in range(target_len):
        start = int(i * step)
        end = int((i + 1) * step)

        # Ensure indices are within bounds and handle empty segments
        if start >= len(feature_series) or start == end:
            # If the segment is empty or out of bounds, take the last available value or NaN
            if len(feature_series) > 0:
                resampled_features.append(feature_series.iloc[-1])
            else:
                resampled_features.append(np.nan)
        else:
            resampled_features.append(feature_series.iloc[start:end].mean())
    return np.array(resampled_features)

# Function to apply DWT and create a dataframe for coefficients and downsampled features
def apply_dwt_and_align_features(group_df, macro_features, transient_features, wavelet='db4'):
    demand_series = group_df['Demand'].values

    # db4 wavelet requires a minimum signal length of 8
    if len(demand_series) < pywt.Wavelet(wavelet).dec_len:
        # Signal too short for decomposition, return empty DataFrames
        return pd.DataFrame(), pd.DataFrame()

    # Perform single-level DWT
    cA, cD = pywt.dwt(demand_series, wavelet)

    num_coeffs = len(cA) # cA and cD will have the same length

    # Downsample macro features
    aligned_macro_features = {}
    for feature in macro_features:
        aligned_macro_features[feature] = downsample_features_simple(group_df[feature], num_coeffs)

    # Downsample transient features
    aligned_transient_features = {}
    for feature in transient_features:
        aligned_transient_features[feature] = downsample_features_simple(group_df[feature], num_coeffs)

    # Create DataFrames for cA and cD targets with aligned features
    # Ensure all arrays have the same length before creating DataFrame
    if not all(len(arr) == num_coeffs for arr in aligned_macro_features.values()):
        print(f"Macro feature length mismatch for Store ID: {group_df['Store ID'].iloc[0]}, Product ID: {group_df['Product ID'].iloc[0]}")
        return pd.DataFrame(), pd.DataFrame()
    if not all(len(arr) == num_coeffs for arr in aligned_transient_features.values()):
        print(f"Transient feature length mismatch for Store ID: {group_df['Store ID'].iloc[0]}, Product ID: {group_df['Product ID'].iloc[0]}")
        return pd.DataFrame(), pd.DataFrame()

    df_cA_group = pd.DataFrame({**aligned_macro_features, 'cA_target': cA})
    df_cD_group = pd.DataFrame({**aligned_transient_features, 'cD_target': cD})

    # Add Store ID and Product ID to both dataframes for identification
    df_cA_group['Store ID'] = group_df['Store ID'].iloc[0]
    df_cA_group['Product ID'] = group_df['Product ID'].iloc[0]
    df_cD_group['Store ID'] = group_df['Store ID'].iloc[0]
    df_cD_group['Product ID'] = group_df['Product ID'].iloc[0]

    return df_cA_group, df_cD_group

# Define features as per the methodology
macro_inputs = ['Price', 'Competitor Pricing', 'Inventory Level']
transient_inputs = ['Promotion', 'Discount', 'Epidemic']

# Add encoded categorical features to the respective lists
seasonality_encoded_cols = [col for col in df_encoded.columns if 'Seasonality_' in col]
weather_encoded_cols = [col for col in df_encoded.columns if 'Weather Condition_' in col]

macro_inputs.extend(seasonality_encoded_cols)
transient_inputs.extend(weather_encoded_cols)

# Initialize empty DataFrames to store the results
all_cA_data = pd.DataFrame()
all_cD_data = pd.DataFrame()

# Group by 'Store ID' and 'Product ID'
grouped = df_encoded.groupby(['Store ID', 'Product ID'])

for (store_id, product_id), group in grouped:
    df_cA_group, df_cD_group = apply_dwt_and_align_features(group, macro_inputs, transient_inputs)

    if not df_cA_group.empty and not df_cD_group.empty:
        all_cA_data = pd.concat([all_cA_data, df_cA_group], ignore_index=True)
        all_cD_data = pd.concat([all_cD_data, df_cD_group], ignore_index=True)

print("Shape of all_cA_data:", all_cA_data.shape)
display(all_cA_data.head())
print("Shape of all_cD_data:", all_cD_data.shape)
display(all_cD_data.head())

## Frequency-Specific Parallel Random Forests

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# --- Model A: RF_Trend (for cA coefficients) ---

# Define features (X_cA) and target (y_cA)
X_cA = all_cA_data.drop(columns=['cA_target', 'Store ID', 'Product ID'])
y_cA = all_cA_data['cA_target']

# Split data into training and testing sets
X_cA_train, X_cA_test, y_cA_train, y_cA_test = train_test_split(X_cA, y_cA, test_size=0.2, random_state=42)

# Initialize and train RandomForestRegressor for cA
rf_trend_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_trend_model.fit(X_cA_train, y_cA_train)

# Make predictions
y_cA_pred = rf_trend_model.predict(X_cA_test)

# Evaluate Model A
mse_cA = mean_squared_error(y_cA_test, y_cA_pred)
r2_cA = r2_score(y_cA_test, y_cA_pred)
print(f"RF_Trend (Model A) - Mean Squared Error (cA): {mse_cA:.4f}")
print(f"RF_Trend (Model A) - R-squared (cA): {r2_cA:.4f}")

# Store test data and predictions for cA for later reconstruction
rf_trend_results = pd.DataFrame({
    'Store ID': all_cA_data.loc[X_cA_test.index, 'Store ID'],
    'Product ID': all_cA_data.loc[X_cA_test.index, 'Product ID'],
    'y_cA_test': y_cA_test,
    'y_cA_pred': y_cA_pred
})


# --- Model B: RF_Spikes (for cD coefficients) ---

# Define features (X_cD) and target (y_cD)
X_cD = all_cD_data.drop(columns=['cD_target', 'Store ID', 'Product ID'])
y_cD = all_cD_data['cD_target']

# Split data into training and testing sets
X_cD_train, X_cD_test, y_cD_train, y_cD_test = train_test_split(X_cD, y_cD, test_size=0.2, random_state=42)

# Initialize and train RandomForestRegressor for cD
rf_spikes_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_spikes_model.fit(X_cD_train, y_cD_train)

# Make predictions
y_cD_pred = rf_spikes_model.predict(X_cD_test)

# Evaluate Model B
mse_cD = mean_squared_error(y_cD_test, y_cD_pred)
r2_cD = r2_score(y_cD_test, y_cD_pred)
print(f"\nRF_Spikes (Model B) - Mean Squared Error (cD): {mse_cD:.4f}")
print(f"RF_Spikes (Model B) - R-squared (cD): {r2_cD:.4f}")

# Store test data and predictions for cD for later reconstruction
rf_spikes_results = pd.DataFrame({
    'Store ID': all_cD_data.loc[X_cD_test.index, 'Store ID'],
    'Product ID': all_cD_data.loc[X_cD_test.index, 'Product ID'],
    'y_cD_test': y_cD_test,
    'y_cD_pred': y_cD_pred
})

print("\nRandom Forest Models Trained and Evaluated.")

## Signal Reconstruction (Inverse DWT)

In [ ]:
import pywt

def reconstruct_demand(df_cA_results, df_cD_results, original_demand_test):
    # Merge cA and cD results on Store ID and Product ID and ensure alignment
    merged_results = pd.merge(df_cA_results, df_cD_results,
                              on=['Store ID', 'Product ID', 'y_cA_test', 'y_cA_pred', 'y_cD_test', 'y_cD_pred'],
                              how='inner')

    # Group by Store ID and Product ID to reconstruct each series
    reconstructed_forecasts = []
    reconstructed_actuals = []

    # It's crucial that the original_demand_test contains the original demand values for the test set indices.
    # We need to map the reconstructed forecasts back to the original time series structure.

    # For simplicity in this step, we will reconstruct based on the predicted cA and cD from the test sets.
    # A more robust approach would involve predicting for future time steps and then reconstructing.
    # Here, we will just demonstrate the reconstruction process on the existing test set predictions.

    for (store_id, product_id), group in merged_results.groupby(['Store ID', 'Product ID']):
        # Get predicted cA and cD for the group
        predicted_cA = group['y_cA_pred'].values
        predicted_cD = group['y_cD_pred'].values

        # Perform Inverse DWT
        reconstructed_y = pywt.idwt(predicted_cA, predicted_cD, 'db4')
        reconstructed_actual = pywt.idwt(group['y_cA_test'].values, group['y_cD_test'].values, 'db4')

        # Add reconstructed data to lists
        reconstructed_forecasts.append(pd.DataFrame({
            'Store ID': store_id,
            'Product ID': product_id,
            'Predicted_Demand_Reconstructed': reconstructed_y
        }))
        reconstructed_actuals.append(pd.DataFrame({
            'Store ID': store_id,
            'Product ID': product_id,
            'Actual_Demand_Reconstructed': reconstructed_actual
        }))

    final_predicted_df = pd.concat(reconstructed_forecasts, ignore_index=True)
    final_actual_df = pd.concat(reconstructed_actuals, ignore_index=True)

    return final_predicted_df, final_actual_df


# Before calling reconstruct_demand, we need to ensure that rf_trend_results and rf_spikes_results have a common index for merging.
# The current way of constructing rf_trend_results and rf_spikes_results uses X_cA_test.index and X_cD_test.index.
# Since train_test_split is called independently for cA and cD, their test indices might not align one-to-one based on original series.
# To fix this, we need to make sure that the split is done in a way that preserves the original series structure for reconstruction.

# Let's re-run the split ensuring we keep track of the original indices from all_cA_data and all_cD_data

# Re-split to preserve original indices for reconstruction
# We will use a combined dataframe to ensure consistent splits across cA and cD data for the same (Store ID, Product ID) group.

# Combine cA and cD data for consistent splitting
# Note: This step assumes that 'all_cA_data' and 'all_cD_data' are already aligned by 'Store ID' and 'Product ID' and index.
# We need to make sure that the test_size refers to the actual groups, not just the rows. This is complex with direct train_test_split.

# A more robust way: iterate through groups, split each group, then combine.

# Let's adjust the splitting logic to ensure proper reconstruction alignment.
# Instead of a global train_test_split, we need to ensure that the cA and cD come from the same original demand series segments.

# For simplicity and to demonstrate the reconstruction, let's assume rf_trend_results and rf_spikes_results have perfectly matching indices and order for now.
# In a real-world scenario, you'd need to ensure the test sets correspond to the same time periods and store/product IDs.

# For the purpose of demonstration, let's try to align them by creating a unique identifier based on original indices, Store ID, and Product ID.

# The current rf_trend_results and rf_spikes_results DataFrames already contain 'Store ID', 'Product ID', 'y_cA_test', 'y_cA_pred', 'y_cD_test', 'y_cD_pred'.
# Let's merge them to ensure we are reconstructing corresponding cA and cD pairs.

# Add a unique identifier for each row in the test sets to ensure merging is correct
rf_trend_results['original_index'] = X_cA_test.index
rf_spikes_results['original_index'] = X_cD_test.index

# Merge the results from RF_Trend and RF_Spikes models based on 'original_index', 'Store ID', and 'Product ID'
# This ensures that for each segment of the original demand series in the test set, we have its predicted cA and cD.
combined_test_predictions = pd.merge(rf_trend_results, rf_spikes_results,
                                     on=['original_index', 'Store ID', 'Product ID'],
                                     suffixes=('_cA', '_cD'))


# Initialize DataFrames to store reconstructed forecasts and actuals
final_reconstructed_forecasts = []
final_reconstructed_actuals = []

# Group by Store ID and Product ID to perform IDWT on each time series segment
for (store_id, product_id), group in combined_test_predictions.groupby(['Store ID', 'Product ID']):
    # Ensure the order of coefficients is preserved for IDWT
    # The order will be based on the 'original_index' if they were sorted before the split,
    # but we need to verify that this group represents a contiguous sequence for DWT.
    # For this exercise, we'll assume the 'original_index' within each group maintains the original time order for IDWT.

    # Sort by original_index to ensure time series order for IDWT
    group = group.sort_values(by='original_index').reset_index(drop=True)

    # Extract predicted and actual cA and cD values
    predicted_cA = group['y_cA_pred'].values
    predicted_cD = group['y_cD_pred'].values
    actual_cA = group['y_cA_test'].values # Corrected column name
    actual_cD = group['y_cD_test'].values # Corrected column name

    # Check if the lengths are valid for IDWT
    # pywt.idwt requires len(cA) == len(cD)
    if len(predicted_cA) > 0 and len(predicted_cD) > 0 and len(predicted_cA) == len(predicted_cD):
        # Reconstruct predicted demand
        reconstructed_predicted_demand = pywt.idwt(predicted_cA, predicted_cD, 'db4')

        # Reconstruct actual demand (from actual cA and cD values for comparison)
        reconstructed_actual_demand = pywt.idwt(actual_cA, actual_cD, 'db4')

        # Append to results
        final_reconstructed_forecasts.append(pd.DataFrame({
            'Store ID': store_id,
            'Product ID': product_id,
            'Reconstructed_Predicted_Demand': reconstructed_predicted_demand
        }))
        final_reconstructed_actuals.append(pd.DataFrame({
            'Store ID': store_id,
            'Product ID': product_id,
            'Reconstructed_Actual_Demand': reconstructed_actual_demand
        }))
    else:
        # Handle cases where lengths might not match or are empty after grouping and filtering
        # This could happen if a group was too short for DWT initially and resulted in empty cA/cD arrays
        print(f"Skipping reconstruction for (Store ID: {store_id}, Product ID: {product_id}) due to invalid coefficient lengths: predicted_cA={len(predicted_cA)}, predicted_cD={len(predicted_cD)}")

# Concatenate all reconstructed dataframes
if final_reconstructed_forecasts:
    final_predicted_df = pd.concat(final_reconstructed_forecasts, ignore_index=True)
    final_actual_df = pd.concat(final_reconstructed_actuals, ignore_index=True)

    print("Shape of final_predicted_df:", final_predicted_df.shape)
    display(final_predicted_df.head())
    print("Shape of final_actual_df:", final_actual_df.shape)
    display(final_actual_df.head())

    # Evaluate the final reconstructed demand forecast
    # Merge the reconstructed predicted and actual demands to calculate overall metrics
    # We need to make sure the merge aligns correctly, assuming the order in each group is preserved.
    # For evaluation, we ideally need a one-to-one mapping between predicted and actual reconstructed values.

    # Simple evaluation by assuming direct correspondence after concatenation, but ideally need original indices/dates
    # This is a simplification; for a robust evaluation, merge on a common time index/identifier.
    # However, since DWT reduces length, direct comparison to original demand is tricky. We compare reconstructed actuals to reconstructed predictions.

    # Ensure the DataFrames are of similar length and correctly aligned for metric calculation.
    # For this exercise, let's assume direct alignment for calculating RMSE/MSE between reconstructed_predicted and reconstructed_actual.
    # A more rigorous approach would involve carefully mapping these back to original time steps if possible.

    # First, let's merge the two reconstructed dataframes on 'Store ID' and 'Product ID' to align them for evaluation
    evaluation_df = pd.merge(final_predicted_df, final_actual_df, on=['Store ID', 'Product ID'])

    # Calculate MSE and R2 for the final reconstructed demand
    final_mse = mean_squared_error(evaluation_df['Reconstructed_Actual_Demand'], evaluation_df['Reconstructed_Predicted_Demand'])
    final_r2 = r2_score(evaluation_df['Reconstructed_Actual_Demand'], evaluation_df['Reconstructed_Predicted_Demand'])

    print(f"\nFinal Reconstructed Demand - Mean Squared Error: {final_mse:.4f}")
    print(f"Final Reconstructed Demand - R-squared: {final_r2:.4f}")
else:
    print("No data available for reconstruction after filtering and grouping.")

### Model Evaluation Metrics

In [ ]:
print(f"\n--- RF_Trend (Model A) Metrics ---")
print(f"Mean Squared Error (cA): {mse_cA:.4f}")
print(f"R-squared (cA): {r2_cA:.4f}")

print(f"\n--- RF_Spikes (Model B) Metrics ---")
print(f"Mean Squared Error (cD): {mse_cD:.4f}")
print(f"R-squared (cD): {r2_cD:.4f}")

print(f"\n--- Final Reconstructed Demand Metrics ---")
print(f"Mean Squared Error: {final_mse:.4f}")
print(f"R-squared: {final_r2:.4f}")

### Interpretation of Results

*   **RF_Trend (Model A) - R-squared (cA): 0.0821**
    *   This indicates that approximately 8.21% of the variance in the low-frequency component (trend) of demand can be explained by the macro-economic features (Price, Competitor Pricing, Inventory Level, Seasonality). This is a very low R-squared, suggesting that Model A is not capturing the trend well or that the chosen macro features are not strong predictors.

*   **RF_Spikes (Model B) - R-squared (cD): -0.0137**
    *   A negative R-squared value means that the model performs worse than simply predicting the mean of the dependent variable. In this case, Model B is not able to explain any variance in the high-frequency component (spikes) of demand using the transient features (Promotion, Discount, Epidemic, Weather Condition) and is essentially performing worse than a baseline model.

*   **Final Reconstructed Demand - R-squared: -0.0343**
    *   The overall reconstructed demand also shows a negative R-squared, which is consistent with the poor performance of the individual models. This suggests that the combined DWT-Random Forest approach, with the current feature set and model configurations, is not effectively forecasting the demand.

### Next Steps for Improvement

Given the low R-squared values, here are some potential areas for improvement:

1.  **Refine Feature Engineering:** Explore more relevant features or create interaction terms. For example, consider lagged demand, moving averages of price/promotions, or more sophisticated seasonality indicators.
2.  **Experiment with Wavelet Parameters:** Try different wavelets (e.g., 'haar', 'sym5') or different decomposition levels. The choice of wavelet and level can significantly impact the separation of trend and seasonality.
3.  **Explore Different Models:** While Random Forests are powerful, other models might be better suited for time series forecasting or for capturing the specific patterns in the cA and cD coefficients. Consider models like: Long Short-Term Memory (LSTM) networks, Prophet, ARIMA variants, or Gradient Boosting Machines (e.g., XGBoost, LightGBM).
4.  **Hyperparameter Tuning:** Systematically tune the hyperparameters of the Random Forest models (e.g., `n_estimators`, `max_depth`, `min_samples_leaf`) using techniques like GridSearchCV or RandomizedSearchCV.
5.  **Cross-Validation:** Implement a time-series specific cross-validation strategy (e.g., rolling origin cross-validation) to get a more robust estimate of model performance.
6.  **Outlier Detection and Handling:** Investigate if outliers in demand or features are disproportionately affecting model training.
7.  **Data Quality:** Re-verify the data quality and ensure all features are correctly preprocessed and scaled if necessary (though tree-based models are less sensitive to scaling).

Which of these areas would you like to explore first?

### Product Ordering Suggestion Dashboard
Now we will scale up the inventory decision logic to compile a complete recommendation sheet across all available Stores and Products.

In [ ]:
import pandas as pd

# Calculate total predicted demand for every Store ID and Product ID over the test period
recommendations = final_predicted_df.groupby(['Store ID', 'Product ID'])['Reconstructed_Predicted_Demand'].sum().reset_index()
recommendations.rename(columns={'Reconstructed_Predicted_Demand': 'Predicted_Demand'}, inplace=True)

# Define inventory parameters (These can be dynamically mapped from a database in real production)
# For demonstration, we assume a uniform baseline for simplicity
recommendations['Current_Inventory'] = 50  # Mock current stock level
recommendations['Safety_Stock'] = 20      # Mock safety stock

# Calculate Suggested Order Quantity
recommendations['Required_Stock'] = recommendations['Predicted_Demand'] + recommendations['Safety_Stock']
recommendations['Suggested_Order_Quantity'] = (recommendations['Required_Stock'] - recommendations['Current_Inventory']).clip(lower=0).round().astype(int)

# Sort recommendations starting with products that need the most stock
recommendations = recommendations.sort_values(by='Suggested_Order_Quantity', ascending=False).reset_index(drop=True)

# Display the top suggestions dashboard
print("--- Top 15 Suggested Product Orders across All Stores ---")
display(recommendations.head(15))

# Save the actionable suggestion sheet to a CSV
recommendations.to_csv('/content/product_order_suggestions.csv', index=False)
print("\nSuccess! The order recommendation dashboard has been saved to '/content/product_order_suggestions.csv'.")

### Actual vs. Predicted Demand Visualization
To visually inspect how closely our reconstructed predictions match the actual sales demand, we can plot the two timelines together for a specific Store and Product.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Choose a specific store and product to visualize
plot_store = 'S001'
plot_product = 'P0001'

# Filter actuals and predictions
actual_series = final_actual_df[
    (final_actual_df['Store ID'] == plot_store) &
    (final_actual_df['Product ID'] == plot_product)
].reset_index(drop=True)

predicted_series = final_predicted_df[
    (final_predicted_df['Store ID'] == plot_store) &
    (final_predicted_df['Product ID'] == plot_product)
].reset_index(drop=True)

# Plot the comparison
plt.figure(figsize=(14, 6))
plt.plot(actual_series['Reconstructed_Actual_Demand'], label='Actual Demand', color='teal', alpha=0.8, linewidth=2)
plt.plot(predicted_series['Reconstructed_Predicted_Demand'], label='Predicted Demand', color='tomato', linestyle='--', linewidth=2)

plt.title(f'Actual vs. Predicted Demand Comparison ({plot_product} at {plot_store})', fontsize=14)
plt.xlabel('Evaluation Time Steps', fontsize=12)
plt.ylabel('Demand Units', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

### Actual vs. Predicted Demand Table
This table aligns the reconstructed actual demand values with our model's predictions to directly observe the exact variance and prediction errors.

In [ ]:
# Merge the actual and predicted series into a single comparison DataFrame
comparison_df = pd.DataFrame({
    'Store ID': actual_series['Store ID'],
    'Product ID': actual_series['Product ID'],
    'Actual Demand': actual_series['Reconstructed_Actual_Demand'].round(2),
    'Predicted Demand': predicted_series['Reconstructed_Predicted_Demand'].round(2)
})

# Calculate the absolute difference/error for each row
comparison_df['Absolute Error'] = (comparison_df['Actual Demand'] - comparison_df['Predicted Demand']).abs().round(2)

# Display the top 20 rows of the comparison table
print("--- Aligned Comparison Table (Actual vs. Predicted) ---")
display(comparison_df.head(20))

### Interactive Forecasting Dashboard
Use the dynamic interface below to forecast daily demand for any custom month, store, and product combination.

In [ ]:
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

def forecast_custom_month(month_year_str, sample_store='S001', sample_product='P0001'):
    try:
        parsed_date = datetime.strptime(month_year_str, '%m-%Y')
    except ValueError:
        print("Invalid format! Please use 'MM-YYYY' (e.g., '05-2024').")
        return None

    year = parsed_date.year
    month = parsed_date.month

    start_date = f"{year}-{month:02d}-01"
    future_dates = pd.date_range(start=start_date, periods=1, freq='M')
    end_date = future_dates[0].strftime('%Y-%m-%d')
    dates_array = pd.date_range(start=start_date, end=end_date, freq='D')
    num_days = len(dates_array)

    spring_flag = 1.0 if month in [3, 4, 5] else 0.0
    summer_flag = 1.0 if month in [6, 7, 8] else 0.0
    winter_flag = 1.0 if month in [12, 1, 2] else 0.0

    baselines = df_encoded.groupby(['Store ID', 'Product ID'])[['Price', 'Competitor Pricing', 'Inventory Level']].mean().reset_index()

    future_records = []
    for _, row in baselines.iterrows():
        s_id = row['Store ID']
        p_id = row['Product ID']
        avg_p = row['Price']
        avg_cp = row['Competitor Pricing']
        avg_inv = row['Inventory Level']

        for day_idx, d in enumerate(dates_array):
            future_records.append({
                'Date': d,
                'Store ID': s_id,
                'Product ID': p_id,
                'Price': avg_p,
                'Competitor Pricing': avg_cp,
                'Inventory Level': avg_inv,
                'Promotion': 1 if day_idx % 7 in [4, 5] else 0,
                'Discount': 10 if day_idx % 7 in [4, 5] else 0,
                'Epidemic': 0,
                'Seasonality_Spring': spring_flag,
                'Seasonality_Summer': summer_flag,
                'Seasonality_Winter': winter_flag,
                'Weather Condition_Rainy': 0.0,
                'Weather Condition_Snowy': 1.0 if (month in [12, 1, 2] and day_idx % 5 == 0) else 0.0,
                'Weather Condition_Sunny': 1.0 if (month in [6, 7, 8] and day_idx % 4 == 0) else 0.0
            })

    custom_features_df = pd.DataFrame(future_records)
    target_coeff_len = len(pywt.dwt(np.zeros(num_days), 'db4')[0])

    cA_records, cD_records = [], []
    for (s_id, p_id), group in custom_features_df.groupby(['Store ID', 'Product ID']):
        aligned_macro = {feat: downsample_features_simple(group[feat], target_coeff_len)
                         for feat in ['Price', 'Competitor Pricing', 'Inventory Level', 'Seasonality_Spring', 'Seasonality_Summer', 'Seasonality_Winter']}
        aligned_transient = {feat: downsample_features_simple(group[feat], target_coeff_len)
                             for feat in ['Promotion', 'Discount', 'Epidemic', 'Weather Condition_Rainy', 'Weather Condition_Snowy', 'Weather Condition_Sunny']}

        df_cA = pd.DataFrame(aligned_macro)
        df_cA['Store ID'], df_cA['Product ID'] = s_id, p_id

        df_cD = pd.DataFrame(aligned_transient)
        df_cD['Store ID'], df_cD['Product ID'] = s_id, p_id

        cA_records.append(df_cA)
        cD_records.append(df_cD)

    cA_df = pd.concat(cA_records, ignore_index=True)
    cD_df = pd.concat(cD_records, ignore_index=True)

    X_cA = cA_df.drop(columns=['Store ID', 'Product ID'])
    X_cD = cD_df.drop(columns=['Store ID', 'Product ID'])

    cA_df['pred_cA'] = rf_trend_model.predict(X_cA)
    cD_df['pred_cD'] = rf_spikes_model.predict(X_cD)

    merged_custom = pd.merge(cA_df[['Store ID', 'Product ID', 'pred_cA']], cD_df[['Store ID', 'Product ID', 'pred_cD']], on=['Store ID', 'Product ID'])

    forecast_records = []
    for (s_id, p_id), group in merged_custom.groupby(['Store ID', 'Product ID']):
        p_cA = group['pred_cA'].values
        p_cD = group['pred_cD'].values

        reconstructed_full = pywt.idwt(p_cA, p_cD, 'db4')
        reconstructed_demand = reconstructed_full[:num_days]

        group_df = pd.DataFrame({
            'Store ID': s_id,
            'Product ID': p_id,
            'Date': dates_array,
            'Predicted_Demand': reconstructed_demand
        })
        forecast_records.append(group_df)

    forecast_df = pd.concat(forecast_records, ignore_index=True)
    print(f"=== Successfully Forecasted Demand for {month_year_str} ===")

    sample_plot_df = forecast_df[(forecast_df['Store ID'] == sample_store) & (forecast_df['Product ID'] == sample_product)]

    plt.figure(figsize=(12, 5))
    plt.plot(sample_plot_df['Date'], sample_plot_df['Predicted_Demand'], label='Forecasted Demand', color='navy', marker='o', linestyle='-')
    plt.title(f"Wavelet-RF Forecast for {month_year_str} (Item {sample_product} at {sample_store})", fontsize=14)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Demand Units', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    return forecast_df

In [ ]:
unique_stores = sorted(df_encoded['Store ID'].unique())
unique_products = sorted(df_encoded['Product ID'].unique())

month_input = widgets.Text(value='05-2024', placeholder='MM-YYYY', description='Month-Year:')
store_dropdown = widgets.Dropdown(options=unique_stores, value=unique_stores[0], description='Store ID:')
product_dropdown = widgets.Dropdown(options=unique_products, value=unique_products[0], description='Product ID:')
button = widgets.Button(description='Generate Forecast', button_style='success', icon='check')
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output(wait=True)
        mm_yyyy = month_input.value.strip()
        store = store_dropdown.value
        product = product_dropdown.value

        forecast_result = forecast_custom_month(mm_yyyy, sample_store=store, sample_product=product)
        if forecast_result is not None:
            specific_df = forecast_result[(forecast_result['Store ID'] == store) & (forecast_result['Product ID'] == product)]
            print(f"\n--- Forecasted Daily Demand Table ({store} - {product}) ---")
            display(specific_df.reset_index(drop=True))

button.on_click(on_button_clicked)

input_box = widgets.VBox([month_input, store_dropdown, product_dropdown, button])
display(widgets.HTML("<h3>Interactive Wavelet-RF Demand Forecasting Dashboard</h3>"))
display(input_box)
display(output_area)